# LR Sweep on Colab (Tiny × 5 LRs)

SVG Scaling Laws Project — LR Sweep reproduction on CUDA

**Runtime**: A100 GPU

**Purpose**: Reproduce Mac Mini (MPS) LR sweep on CUDA to isolate environment differences

**Estimated time**: ~25 min (5 runs × ~5 min each)

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip -q /content/drive/MyDrive/SVGScalingLawProject/svg-scaling-project.zip -d /content/

In [ ]:
!pip install -q pyyaml mup

## 2. Verify GPU

In [4]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

!ls /content/svg-scaling-project/data/tokenized/

CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB
test.bin  tokenize_stats.json  train.bin  val.bin


## 3. Run LR Sweep (Tiny × 5 LRs)

In [5]:
import subprocess, time, json

lrs = ['3e-5', '1e-4', '3e-4', '1e-3', '3e-3']
base_dir = '/content/svg-scaling-project'
results = []

for lr in lrs:
    output_dir = f'{base_dir}/results/runs/lr_sweep_colab/tiny_lr_{lr}'
    print(f'\n{"=" * 60}')
    print(f'Training Tiny with LR={lr}')
    print(f'{"=" * 60}')
    t0 = time.time()

    result = subprocess.run([
        'python', f'{base_dir}/src/train.py',
        '--config', f'{base_dir}/configs/tiny.yaml',
        '--learning-rate', lr,
        '--output-dir', output_dir,
        '--device', 'cuda',
    ], cwd=base_dir)

    elapsed = time.time() - t0
    status = 'OK' if result.returncode == 0 else f'FAIL (exit {result.returncode})'
    print(f'\n→ LR={lr}: {status}, {elapsed/60:.1f} min')

    summary_path = f'{output_dir}/summary.json'
    try:
        with open(summary_path) as f:
            s = json.load(f)
        results.append({'lr': lr, **s})
        print(f'  best_val_loss={s["best_val_loss"]:.4f}, final_ppl={s["final_val_ppl"]:.2f}')
    except FileNotFoundError:
        print(f'  (no summary.json)')

print(f'\n{"=" * 60}')
print('LR Sweep Complete!')
print(f'{"=" * 60}')
print(f'{"LR":>8s}  {"Best Val Loss":>14s}  {"Final PPL":>10s}')
print(f'{"-"*8}  {"-"*14}  {"-"*10}')
for r in results:
    print(f'{r["lr"]:>8s}  {r["best_val_loss"]:>14.4f}  {r["final_val_ppl"]:>10.2f}')


Training Tiny with LR=3e-5

→ LR=3e-5: OK, 2.7 min
  best_val_loss=4.5071, final_ppl=92.68

Training Tiny with LR=1e-4

→ LR=1e-4: OK, 2.6 min
  best_val_loss=3.9589, final_ppl=53.44

Training Tiny with LR=3e-4

→ LR=3e-4: OK, 2.6 min
  best_val_loss=3.4348, final_ppl=31.12

Training Tiny with LR=1e-3

→ LR=1e-3: OK, 2.6 min
  best_val_loss=3.1302, final_ppl=23.24

Training Tiny with LR=3e-3

→ LR=3e-3: OK, 2.6 min
  best_val_loss=2.9737, final_ppl=19.56

LR Sweep Complete!
      LR   Best Val Loss   Final PPL
--------  --------------  ----------
    3e-5          4.5071       92.68
    1e-4          3.9589       53.44
    3e-4          3.4348       31.12
    1e-3          3.1302       23.24
    3e-3          2.9737       19.56


## 4. Compare with Mac Mini Results

In [6]:
# Mac Mini (MPS) results for comparison
mac_mini_results = {
    '3e-5': 4.4753,
    '1e-4': 3.9951,
    '3e-4': 3.3045,
    '1e-3': 3.0930,
    '3e-3': 3.0891,
}

print(f'{"LR":>8s}  {"Mac Mini (MPS)":>14s}  {"Colab (CUDA)":>14s}  {"Diff":>8s}')
print(f'{"-"*8}  {"-"*14}  {"-"*14}  {"-"*8}')
for r in results:
    lr = r['lr']
    colab_loss = r['best_val_loss']
    mac_loss = mac_mini_results.get(lr, float('nan'))
    diff = colab_loss - mac_loss
    print(f'{lr:>8s}  {mac_loss:>14.4f}  {colab_loss:>14.4f}  {diff:>+8.4f}')

      LR  Mac Mini (MPS)    Colab (CUDA)      Diff
--------  --------------  --------------  --------
    3e-5          4.4753          4.5071   +0.0318
    1e-4          3.9951          3.9589   -0.0362
    3e-4          3.3045          3.4348   +0.1303
    1e-3          3.0930          3.1302   +0.0372
    3e-3          3.0891          2.9737   -0.1154


## 5. Save Results to Drive

In [7]:
import shutil, os

drive_dest = '/content/drive/MyDrive/SVGScalingLawProject/lr_sweep_colab_results'
os.makedirs(drive_dest, exist_ok=True)

for lr in lrs:
    src_dir = f'{base_dir}/results/runs/lr_sweep_colab/tiny_lr_{lr}'
    dst_dir = f'{drive_dest}/tiny_lr_{lr}'
    os.makedirs(dst_dir, exist_ok=True)
    for fname in ['summary.json', 'training_log.json']:
        src = f'{src_dir}/{fname}'
        if os.path.exists(src):
            shutil.copy2(src, f'{dst_dir}/{fname}')
            print(f'Copied tiny_lr_{lr}/{fname}')

print('\nAll results saved to Drive.')

Copied tiny_lr_3e-5/summary.json
Copied tiny_lr_3e-5/training_log.json
Copied tiny_lr_1e-4/summary.json
Copied tiny_lr_1e-4/training_log.json
Copied tiny_lr_3e-4/summary.json
Copied tiny_lr_3e-4/training_log.json
Copied tiny_lr_1e-3/summary.json
Copied tiny_lr_1e-3/training_log.json
Copied tiny_lr_3e-3/summary.json
Copied tiny_lr_3e-3/training_log.json

All results saved to Drive.
